In [ ]:
# ===========================================================================
# Install / upgrade all dependencies
#   Run this cell first, then RESTART THE RUNTIME before running Cell 2+
# ===========================================================================
import subprocess, sys

# Remove stale wheels to avoid conflicts
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "torch", "torchvision", "torchaudio",
                "transformers", "peft", "trl", "accelerate", "bitsandbytes"],
               capture_output=True)

# PyTorch with CUDA 12.4 (matches Colab's 2025 driver)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu124"],
               check=True)

# Pin the same library versions used in QLoRA experiments for fair comparison
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.48.1",
                "accelerate==1.3.0",
                "peft==0.14.0",
                "trl==0.15.2",
                "bitsandbytes>=0.43.0",   # kept for env parity; not used for quant
                "datasets>=2.18.0",
                "scikit-learn",
                "pandas",
                "numpy",
                "sentencepiece",
                "protobuf"],
               check=True)

print("\u2705  All packages installed — RESTART THE RUNTIME NOW, then run Cell 2+")

✅  All packages installed — RESTART THE RUNTIME NOW, then run Cell 2+


In [ ]:
# ===========================================================================
# Mount Google Drive & HuggingFace login
# ===========================================================================
from google.colab import drive
drive.mount("/content/drive")

from huggingface_hub import login
HF_TOKEN = "hf_*************************"   
login(HF_TOKEN)
print("\u2705  Drive mounted & HF login complete.")

Mounted at /content/drive
✅  Drive mounted & HF login complete.


In [ ]:
# ===========================================================================
# Imports & reproducibility seeds
# ===========================================================================
import os, json, random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
)
# NOTE: No BitsAndBytesConfig import needed — this is full LoRA (no quantisation)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("\u2705  Imports done.")
print("    GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch to GPU runtime!")
print("    CUDA memory available:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

✅  Imports done.
    GPU: NVIDIA A100-SXM4-80GB
    CUDA memory available: 85.1 GB


In [ ]:
# ===========================================================================
# Project paths
#
#   New LoRA v2 directory — does NOT overwrite QLoRA runs.
#   Structure mirrors the QLoRA layout for consistent evaluation scripts.
# ===========================================================================
BASE_MODEL_ID   = "microsoft/Phi-4-mini-instruct"
HF_DATASET_ID   = "darklord1611/legal_citations"

# LoRA v2 dedicated directory 
PROJECT_DIR     = "/content/drive/MyDrive/phi4_thesis_LoRA"
SPLITS_DIR      = os.path.join(PROJECT_DIR, "splits")
LORA_DIR        = os.path.join(PROJECT_DIR, "lora_model")
METRICS_DIR     = os.path.join(PROJECT_DIR, "metrics")
PREDICTIONS_DIR = os.path.join(PROJECT_DIR, "predictions")

for folder in [PROJECT_DIR, SPLITS_DIR, LORA_DIR, METRICS_DIR, PREDICTIONS_DIR]:
    os.makedirs(folder, exist_ok=True)

print("\u2705  Directories ready under:", PROJECT_DIR)
print("    lora_model  :", LORA_DIR)
print("    splits      :", SPLITS_DIR)
print("    metrics     :", METRICS_DIR)
print("    predictions :", PREDICTIONS_DIR)

✅  Directories ready under: /content/drive/MyDrive/phi4_thesis_LoRA
    lora_model  : /content/drive/MyDrive/phi4_thesis_LoRA/lora_model
    splits      : /content/drive/MyDrive/phi4_thesis_LoRA/splits
    metrics     : /content/drive/MyDrive/phi4_thesis_LoRA/metrics
    predictions : /content/drive/MyDrive/phi4_thesis_LoRA/predictions


In [ ]:
# ===========================================================================
# Load dataset from HuggingFace & inspect
# ===========================================================================
print(f"Loading dataset: {HF_DATASET_ID} ...")
raw = load_dataset(HF_DATASET_ID)

print("\nAvailable splits :", list(raw.keys()))
print("Train rows       :", len(raw["train"]))
print("Test  rows       :", len(raw["test"]))
print("Columns          :", raw["train"].column_names)
print("\nSample row:")
print(raw["train"][0])

Loading dataset: darklord1611/legal_citations ...


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/58.7M [00:00<?, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21237 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3748 [00:00<?, ? examples/s]


Available splits : ['train', 'test']
Train rows       : 21237
Test  rows       : 3748
Columns          : ['Unnamed: 0', 'case_id', 'case_outcome', 'case_title', 'case_text']

Sample row:
{'Unnamed: 0': 21382, 'case_id': 'Case21569', 'case_outcome': 'distinguished', 'case_title': 'Housing Guarantee Fund Ltd v Yusef [1991] 2 VR 17', 'case_text': "The second issue to which I referred, can best be considered by reference to Housing Guarantee Fund Ltd v Yusef [1991] 2 VR 17 (' HGFL v Yusef ')."}


In [ ]:
# ===========================================================================
# Convert to DataFrames & normalise column names
# ===========================================================================
TEXT_COL  = "case_text"
LABEL_COL = "case_outcome"

train_df_raw = raw["train"].to_pandas()
test_df_raw  = raw["test"].to_pandas()

# Defensive column normalisation
for df in [train_df_raw, test_df_raw]:
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Validate required columns exist
for col in [TEXT_COL, LABEL_COL]:
    assert col in train_df_raw.columns, \
        f"Column '{col}' not found. Available: {train_df_raw.columns.tolist()}"

train_df_raw = train_df_raw[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)
test_df_raw  = test_df_raw[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)

for df in [train_df_raw, test_df_raw]:
    df[TEXT_COL]  = df[TEXT_COL].astype(str).str.strip()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()

print("Raw train shape:", train_df_raw.shape)
print("Raw test  shape:", test_df_raw.shape)
print("\nLabel distribution (train):")
print(train_df_raw[LABEL_COL].value_counts().to_string())

Raw train shape: (21083, 2)
Raw test  shape: (3726, 2)

Label distribution (train):
case_outcome
cited            10296
referred to       3727
applied           2051
followed          1923
considered        1449
discussed          860
distinguished      505
approved            97
related             91
affirmed            84


In [ ]:
# ===========================================================================
#  Class-imbalance correction
#
#   Identical balancing strategy to QLoRA v2 for a fair comparison:
#     MIN_LABEL_COUNT = 100   → drop rare labels (insufficient signal)
#     MAX_LABEL_COUNT = 5000  → cap dominant labels (same as QLoRA v2)
# ===========================================================================
MIN_LABEL_COUNT = 100
MAX_LABEL_COUNT = 5000   # Same cap as QLoRA v2

label_counts = train_df_raw[LABEL_COL].value_counts()
print("── Before balancing (train label counts) ──")
print(label_counts.to_string())

# 1) Remove labels below minimum threshold
valid_labels = label_counts[label_counts >= MIN_LABEL_COUNT].index.tolist()
removed      = label_counts[label_counts <  MIN_LABEL_COUNT].index.tolist()
print(f"\nLabels kept   (>={MIN_LABEL_COUNT} rows): {valid_labels}")
print(f"Labels removed (<{MIN_LABEL_COUNT} rows): {removed}")

train_df_filtered = train_df_raw[train_df_raw[LABEL_COL].isin(valid_labels)].copy()

# 2) Cap oversized labels with deterministic random undersampling
def cap_label_group(group, max_count, seed):
    if len(group) > max_count:
        return group.sample(n=max_count, random_state=seed)
    return group

train_df_balanced = (
    train_df_filtered
    .groupby(LABEL_COL, group_keys=False)
    .apply(lambda g: cap_label_group(g, MAX_LABEL_COUNT, SEED))
    .reset_index(drop=True)
)

print("\n── After balancing (train label counts) ──")
print(train_df_balanced[LABEL_COL].value_counts().to_string())
print("\nBalanced train shape:", train_df_balanced.shape)

LABELS = sorted(train_df_balanced[LABEL_COL].unique().tolist())
print("\nFinal label set:", LABELS)

── Before balancing (train label counts) ──
case_outcome
cited            10296
referred to       3727
applied           2051
followed          1923
considered        1449
discussed          860
distinguished      505
approved            97
related             91
affirmed            84

Labels kept   (>=100 rows): ['cited', 'referred to', 'applied', 'followed', 'considered', 'discussed', 'distinguished']
Labels removed (<100 rows): ['approved', 'related', 'affirmed']

── After balancing (train label counts) ──
case_outcome
cited            5000
referred to      3727
applied          2051
followed         1923
considered       1449
discussed         860
distinguished     505

Balanced train shape: (15515, 2)

Final label set: ['applied', 'cited', 'considered', 'discussed', 'distinguished', 'followed', 'referred to']


/tmp/ipykernel_5708/3810697033.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: cap_label_group(g, MAX_LABEL_COUNT, SEED))


In [ ]:
# ===========================================================================
# Split HF test split → validation + test (50 / 50, stratified)
#
#   Identical splitting strategy to QLoRA v2 — same SEED, same ratios.
#   The test set is NOT rebalanced so evaluation reflects natural distribution.
# ===========================================================================
test_df_filtered = test_df_raw[test_df_raw[LABEL_COL].isin(LABELS)].reset_index(drop=True)

label_counts_test = test_df_filtered[LABEL_COL].value_counts()
can_stratify      = label_counts_test.min() >= 2

val_df, test_df = train_test_split(
    test_df_filtered,
    test_size=0.50,
    random_state=SEED,
    stratify=test_df_filtered[LABEL_COL] if can_stratify else None,
)
val_df  = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape      :", train_df_balanced.shape)
print("Validation shape :", val_df.shape)
print("Test shape       :", test_df.shape)
print("\nValidation label distribution:")
print(val_df[LABEL_COL].value_counts().to_string())
print("\nTest label distribution:")
print(test_df[LABEL_COL].value_counts().to_string())

# Save splits
train_df_balanced.to_csv(os.path.join(SPLITS_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(SPLITS_DIR,  "val.csv"),             index=False)
test_df.to_csv(os.path.join(SPLITS_DIR, "test.csv"),            index=False)
with open(os.path.join(SPLITS_DIR, "labels.json"), "w") as f:
    json.dump(LABELS, f, indent=2)

print("\n\u2705  Splits saved to:", SPLITS_DIR)

Train shape      : (15515, 2)
Validation shape : (1836, 2)
Test shape       : (1836, 2)

Validation label distribution:
case_outcome
cited            907
referred to      318
applied          193
followed         165
considered       125
discussed         79
distinguished     49

Test label distribution:
case_outcome
cited            907
referred to      318
applied          194
followed         164
considered       125
discussed         79
distinguished     49

✅  Splits saved to: /content/drive/MyDrive/phi4_thesis_LoRA/splits


In [ ]:
# ===========================================================================
# Load tokenizer
# ===========================================================================
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
)
tokenizer.model_max_length = 2048

# Phi-4-mini-instruct has no dedicated pad token; use eos to avoid shape issues
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"   # right-pad during training

print("\u2705  Tokenizer loaded.")
print("    Vocab size :", tokenizer.vocab_size)
print("    Pad token  :", repr(tokenizer.pad_token))
print("    EOS token  :", repr(tokenizer.eos_token))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

✅  Tokenizer loaded.
    Vocab size : 200019
    Pad token  : '<|endoftext|>'
    EOS token  : '<|endoftext|>'


In [ ]:
# ===========================================================================
# Build prompt / chat-template formatting function
#
#   System prompt matches QLoRA v2 exactly so the fine-tuning objective
#   is identical — only the quantisation differs.
# ===========================================================================
SYSTEM_MSG = (
    "You are a legal assistant specialising in case-outcome classification. "
    "Analyse the provided case text and return exactly one label from the "
    "allowed list — nothing else."
)

LABEL_STR = ", ".join(LABELS)

def format_example(example):
    """
    Wraps a single row into Phi-4-mini-instruct chat format.
    The assistant turn contains ONLY the normalised label (ground truth).
    """
    messages = [
        {"role": "system",    "content": SYSTEM_MSG},
        {
            "role": "user",
            "content": (
                f"Case Text:\n{example[TEXT_COL]}\n\n"
                f"Allowed labels: {LABEL_STR}.\n"
                "What is the outcome of this case? Return only the label."
            ),
        },
        {
            "role": "assistant",
            "content": str(example[LABEL_COL]).strip().lower(),
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

In [ ]:
# ===========================================================================
#  Create HuggingFace Dataset objects & apply template
# ===========================================================================
raw_ds = DatasetDict({
    "train":      Dataset.from_pandas(train_df_balanced),
    "validation": Dataset.from_pandas(val_df),
})

processed_train = raw_ds["train"].map(
    format_example,
    remove_columns=raw_ds["train"].column_names,
    desc="Formatting train",
)
processed_val = raw_ds["validation"].map(
    format_example,
    remove_columns=raw_ds["validation"].column_names,
    desc="Formatting validation",
)

print("Processed train size :", len(processed_train))
print("Processed val size   :", len(processed_val))
print("\n── Sample formatted text (first 800 chars) ──")
print(processed_train[0]["text"][:800])

Formatting train:   0%|          | 0/15515 [00:00<?, ? examples/s]

Formatting validation:   0%|          | 0/1836 [00:00<?, ? examples/s]

Processed train size : 15515
Processed val size   : 1836

── Sample formatted text (first 800 chars) ──
<|system|>You are a legal assistant specialising in case-outcome classification. Analyse the provided case text and return exactly one label from the allowed list — nothing else.<|end|><|user|>Case Text:
It is sufficient to note that intermediate courts have repeatedly been willing to hold that such a duty exists. However, as noted earlier I was referred to only one decision where the existence of such a duty (express or implied) was held to have been breached: Pacific Brands Sport &amp; Leisure Pty Ltd v Underworks Pty Ltd [2005] FCA 288 at [66] , a decision which Gyles J subsequently described as ' adventurous ': Council of the City of Sydney v Goldspar Australia Pty Ltd (2006) 230 ALR 437 at [166]. Two things should be noted about the decision at first instance. First, the breach of suc


In [ ]:
# ===========================================================================
# Load model in full bfloat16 (NO quantisation)
#
#   LoRA vs QLoRA — the key difference is here:
#     QLoRA: model loaded in 4-bit NF4 via BitsAndBytesConfig
#     LoRA:  model loaded in full bfloat16 — higher fidelity, more VRAM
#
#   Phi-4-mini-instruct (~3.8B params) in bfloat16:
#     Model weights   : ~7.6 GB
#     LoRA adapters   : ~0.4 GB  (r=32, all-linear)
#     Activations/grad: ~8-12 GB (with gradient checkpointing)
#     Total (est.)    : ~18-22 GB  →  fits on A100 40GB
#
#   NOTE: We do NOT call prepare_model_for_kbit_training — that function
#   is only for quantised (k-bit) models and would cause errors here.
#   Gradient checkpointing is enabled directly on the model.
# ===========================================================================
print(f"Loading {BASE_MODEL_ID} in bfloat16 (full precision LoRA) ...")
print("This may take 2-3 minutes on first load.")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,      # full bfloat16 — no quantisation
    device_map="auto",               # auto-shards across available GPUs
    trust_remote_code=True,
    use_cache=False,                 # required for gradient checkpointing
    attn_implementation="eager",     # safest for Phi-4-mini on Colab
)

# Enable gradient checkpointing directly (no kbit wrapper needed)
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# Confirm memory usage after load
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
print(f"\n\u2705  Model loaded in bfloat16.")
print(f"    Total parameters  : {sum(p.numel() for p in model.parameters()):,}")
print(f"    Trainable (before LoRA): {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"    VRAM allocated    : {allocated:.1f} GB")
print(f"    VRAM reserved     : {reserved:.1f} GB")

Loading microsoft/Phi-4-mini-instruct in bfloat16 (full precision LoRA) ...
This may take 2-3 minutes on first load.


config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]


✅  Model loaded in bfloat16.
    Total parameters  : 3,836,021,760
    Trainable (before LoRA): 3,836,021,760
    VRAM allocated    : 7.7 GB
    VRAM reserved     : 7.7 GB


In [ ]:
# ===========================================================================
#  LoRA / PEFT config
#
#   Identical to QLoRA v2 for a controlled comparison:
#     r=32, alpha=64 → capacity matches QLoRA v2
#     target_modules="all-linear" → same layers adapted
#     dropout=0.05, bias="none"  → same regularisation
#
#   SFTTrainer will call get_peft_model() internally when peft_config
#   is passed — we do NOT need to wrap the model manually.
# ===========================================================================
peft_config = LoraConfig(
    r=32,                        # adapter rank — matches QLoRA v2
    lora_alpha=64,               # scaling factor (alpha/r = 2)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",  # adapts every linear layer in Phi-4-mini
)

print("\u2705  LoRA config ready.")
print(f"    r            : {peft_config.r}")
print(f"    alpha        : {peft_config.lora_alpha}")
print(f"    dropout      : {peft_config.lora_dropout}")
print(f"    target_modules: {peft_config.target_modules}")

✅  LoRA config ready.
    r            : 32
    alpha        : 64
    dropout      : 0.05
    target_modules: all-linear


In [ ]:
# ===========================================================================
# SFT training config
#
#   Matches QLoRA v2 training hyperparameters exactly:
#     num_train_epochs = 1    (single pass, Colab-safe)
#     learning_rate = 2e-4    (same as QLoRA v2)
#     batch size / accum / LR schedule — all identical
#
#   The only change:
#     optim = "adamw_torch"   (replaces paged_adamw_8bit which is kbit-specific)
# ===========================================================================
_PER_DEVICE_BATCH = 2
_GRAD_ACCUM       = 4
_EFFECTIVE_BATCH  = _PER_DEVICE_BATCH * _GRAD_ACCUM   # = 8

sft_config = SFTConfig(
    #  Output
    output_dir=LORA_DIR,

    # Precision
    bf16=True,

    #  Epochs & steps
    num_train_epochs=1,    # 1 full epoch — mirrors QLoRA v2
    max_steps=-1,          # honour num_train_epochs

    # Batch & gradient accumulation
    per_device_train_batch_size=_PER_DEVICE_BATCH,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=_GRAD_ACCUM,   # effective batch = 8

    #  Learning rate schedule
    learning_rate=2e-4,          # same as QLoRA v2
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    #  Memory (gradient checkpointing already enabled on the model above) ──
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    #  Optimizer
    # paged_adamw_8bit is only for kbit (quantised) training.
    # For full LoRA we use standard adamw_torch.
    optim="adamw_torch",

    # Logging
    logging_strategy="steps",
    logging_steps=20,
    report_to="none",

    # Evaluation
    do_eval=True,
    eval_strategy="steps",
    eval_steps=300,

    # Checkpointing
    save_strategy="steps",
    save_steps=300,
    save_total_limit=2,

    # Dataset
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
    remove_unused_columns=True,

    # Best-model restoration
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

print("\u2705  SFTConfig ready.")
print(f"   Epochs       : {sft_config.num_train_epochs}")
print(f"   LR           : {sft_config.learning_rate}")
print(f"   Optimizer    : {sft_config.optim}")
print(f"   Eff. batch   : {_PER_DEVICE_BATCH} x {_GRAD_ACCUM} = {_EFFECTIVE_BATCH}")
print(f"   Eval every   : {sft_config.eval_steps} steps")

✅  SFTConfig ready.
   Epochs       : 1
   LR           : 0.0002
   Optimizer    : OptimizerNames.ADAMW_TORCH
   Eff. batch   : 2 x 4 = 8
   Eval every   : 300 steps


In [ ]:
# ===========================================================================
# Build SFTTrainer & start training
# ===========================================================================
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    peft_config=peft_config,
    train_dataset=processed_train,
    eval_dataset=processed_val,
    processing_class=tokenizer,   # replaces deprecated 'tokenizer=' in trl>=0.12
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters after LoRA : {trainable:,}")
print(f"Total parameters                : {total:,}")
print(f"Trainable %                     : {100 * trainable / total:.2f}%")

print("\n\U0001f680  Starting LoRA v2 training ...")
train_result  = trainer.train()
train_metrics = train_result.metrics

print("\n── Training metrics ──")
print(json.dumps(train_metrics, indent=2))

Converting train dataset to ChatML:   0%|          | 0/15515 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/15515 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15515 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2119 > 2048). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/15515 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/1836 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1836 [00:00<?, ? examples/s]

Trainable parameters after LoRA : 46,137,344
Total parameters                : 3,882,159,104
Trainable %                     : 1.19%

🚀  Starting LoRA v2 training ...


Step,Training Loss,Validation Loss
300,1.825100,1.787644
600,1.733200,1.715071
900,1.665800,1.651787
1200,1.633800,1.606162
1500,1.596600,1.574669
1800,1.546900,1.562467



── Training metrics ──
{
  "train_runtime": 6242.5784,
  "train_samples_per_second": 2.485,
  "train_steps_per_second": 0.311,
  "total_flos": 2.378923649630331e+17,
  "train_loss": 1.70464348461039
}


In [ ]:
# ===========================================================================
#  Save model adapter, tokenizer & metrics
#
#   save_model() saves only the LoRA adapter weights (small delta),
#   NOT the full bfloat16 base model (which stays on HuggingFace Hub).
# ===========================================================================
trainer.save_model(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

# Save label list alongside adapter so the evaluation script is self-contained
with open(os.path.join(LORA_DIR,   "labels.json"), "w") as f:
    json.dump(LABELS, f, indent=2)
with open(os.path.join(SPLITS_DIR, "labels.json"), "w") as f:
    json.dump(LABELS, f, indent=2)

# Persist training metrics for the comparison table
with open(os.path.join(METRICS_DIR, "train_metrics.json"), "w") as f:
    json.dump(train_metrics, f, indent=2)

print("\u2705  LoRA adapter saved to:", LORA_DIR)
print("   Files in LORA_DIR:", os.listdir(LORA_DIR))

✅  LoRA adapter saved to: /content/drive/MyDrive/phi4_thesis_LoRA/lora_model
   Files in LORA_DIR: ['checkpoint-1800', 'checkpoint-1939', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'training_args.bin', 'labels.json']


In [ ]:
# ===========================================================================
#  Final validation-loss evaluation & save
# ===========================================================================
eval_metrics = trainer.evaluate()
eval_metrics["eval_samples"] = len(processed_val)

print("\n── Validation metrics ──")
print(json.dumps(eval_metrics, indent=2))

with open(os.path.join(METRICS_DIR, "eval_loss_metrics.json"), "w") as f:
    json.dump(eval_metrics, f, indent=2)

print("\n\u2705  LoRA v2 fine-tuning complete.")
print("    LoRA adapter  :", LORA_DIR)
print("    Splits        :", SPLITS_DIR)
print("    Metrics       :", METRICS_DIR)
print("\nNext step: run the evaluation / inference script on the held-out test set.")
print("Final comparison table: base model (zero-shot) | QLoRA v2 | QLoRA v3 | LoRA v2 | LoRA v3")


── Validation metrics ──
{
  "eval_loss": 1.5624667406082153,
  "eval_runtime": 151.9089,
  "eval_samples_per_second": 12.086,
  "eval_steps_per_second": 6.043,
  "eval_samples": 1836
}

✅  LoRA v2 fine-tuning complete.
    LoRA adapter  : /content/drive/MyDrive/phi4_thesis_LoRA/lora_model
    Splits        : /content/drive/MyDrive/phi4_thesis_LoRA/splits
    Metrics       : /content/drive/MyDrive/phi4_thesis_LoRA/metrics

Next step: run the evaluation / inference script on the held-out test set.
Final comparison table: base model (zero-shot) | QLoRA v2 | QLoRA v3 | LoRA v2 | LoRA v3


In [ ]:

#   Release runtime when done to avoid Colab timeout

from google.colab import runtime
runtime.unassign()